In [1]:
!pip install kagglehub -q
import kagglehub

path = kagglehub.dataset_download("behrad3d/nasa-cmaps")
print(path)   # 파일들이 풀린 폴더 경로

Using Colab cache for faster access to the 'nasa-cmaps' dataset.
/kaggle/input/nasa-cmaps


In [2]:
import os
print(os.listdir(path))

['.nfs000000009da87b9a0000003c', 'CMaps']


In [3]:
import os
cmaps_path = os.path.join(path, "CMaps")
print(os.listdir(cmaps_path))

['RUL_FD002.txt', 'test_FD003.txt', 'Damage Propagation Modeling.pdf', 'readme.txt', 'train_FD003.txt', 'test_FD004.txt', 'train_FD004.txt', 'x.txt', 'test_FD002.txt', 'train_FD001.txt', 'train_FD002.txt', 'RUL_FD001.txt', 'RUL_FD004.txt', 'RUL_FD003.txt', 'test_FD001.txt']


In [18]:
import pandas as pd
train_df = pd.read_csv('/kaggle/input/nasa-cmaps/CMaps/train_FD001.txt', sep=r"\s+", header=None)
train_df = train_df.iloc[:, :26]
test_df = pd.read_csv('/kaggle/input/nasa-cmaps/CMaps/test_FD001.txt', sep=r"\s+", header=None)
test_df = test_df.iloc[:, :26]

col_names = ['unit', 'cycle'] + [f"op{i}" for i in range(1,4)] + [f"s{i}" for i in range(1,22)]
train_df.columns = col_names
test_df.columns = col_names

print('엔진 수:', train_df['unit'].nunique())
print('엔진 별 사이클 수:\n', train_df.groupby('unit')['cycle'].max().describe())
train_df.head()

엔진 수: 100
엔진 별 사이클 수:
 count    100.000000
mean     206.310000
std       46.342749
min      128.000000
25%      177.000000
50%      199.000000
75%      229.250000
max      362.000000
Name: cycle, dtype: float64


,unit,cycle,op1,op2,op3,s1,s2,s3,s4,s5,...,s12,s13,s14,s15,s16,s17,s18,s19,s20,s21
0,1,1,-0.0007,-0.0004,100.0,518.67,641.82,1589.70,1400.60,14.62,...,521.66,2388.02,8138.62,8.4195,0.03,392,2388,100.0,39.06,23.4190
1,1,2,0.0019,-0.0003,100.0,518.67,642.15,1591.82,1403.14,14.62,...,522.28,2388.07,8131.49,8.4318,0.03,392,2388,100.0,39.00,23.4236
2,1,3,-0.0043,0.0003,100.0,518.67,642.35,1587.99,1404.20,14.62,...,522.42,2388.03,8133.23,8.4178,0.03,390,2388,100.0,38.95,23.3442
3,1,4,0.0007,0.0000,100.0,518.67,642.35,1582.79,1401.87,14.62,...,522.86,2388.08,8133.83,8.3682,0.03,392,2388,100.0,38.88,23.3739
4,1,5,-0.0019,-0.0002,100.0,518.67,642.37,1582.85,1406.22,14.62,...,522.19,2388.04,8133.80,8.4294,0.03,393,2388,100.0,38.90,23.4044


In [30]:
train_df['RUL'] = train_df.groupby('unit')['cycle'].transform('max') - train_df['cycle']
train_df['RUL'] = train_df['RUL'].clip(upper=125)
train_df.head()

,unit,cycle,op1,op2,op3,s1,s2,s3,s4,s5,...,s13,s14,s15,s16,s17,s18,s19,s20,s21,RUL
0,1,1,-0.0007,-0.0004,100.0,518.67,0.183735,0.406802,0.309757,14.62,...,0.205882,0.199608,0.363986,0.03,0.333333,2388,100.0,0.713178,0.724662,125
1,1,2,0.0019,-0.0003,100.0,518.67,0.283133,0.453019,0.352633,14.62,...,0.279412,0.162813,0.411312,0.03,0.333333,2388,100.0,0.666667,0.731014,125
2,1,3,-0.0043,0.0003,100.0,518.67,0.343373,0.369523,0.370527,14.62,...,0.220588,0.171793,0.357445,0.03,0.166667,2388,100.0,0.627907,0.621375,125
3,1,4,0.0007,0.0000,100.0,518.67,0.343373,0.256159,0.331195,14.62,...,0.294118,0.174889,0.166603,0.03,0.333333,2388,100.0,0.573643,0.662386,125
4,1,5,-0.0019,-0.0002,100.0,518.67,0.349398,0.257467,0.404625,14.62,...,0.235294,0.174734,0.402078,0.03,0.416667,2388,100.0,0.589147,0.704502,125


In [33]:
sensor_cols = [f"s{i}" for i in range(1,22)]
train_stds = train_df[sensor_cols].std().sort_values()
print(train_stds)
useless = train_stds[train_stds < 0.001].index.tolist()
print(useless)
used_sensors = [c for c in sensor_cols if c not in useless]
print("쓸 센서:", used_sensors)

s19    0.000000e+00
s18    0.000000e+00
s16    1.556432e-14
s10    4.660829e-13
s5     3.394700e-12
s1     6.537152e-11
s14    9.844244e-02
s9     9.908857e-02
s13    1.057631e-01
s8     1.075538e-01
s17    1.290636e-01
s3     1.336636e-01
s6     1.388985e-01
s20    1.401135e-01
s7     1.425269e-01
s15    1.443056e-01
s21    1.494765e-01
s2     1.506185e-01
s4     1.519346e-01
s12    1.572609e-01
s11    1.589806e-01
dtype: float64
['s19', 's18', 's16', 's10', 's5', 's1']
쓸 센서: ['s2', 's3', 's4', 's6', 's7', 's8', 's9', 's11', 's12', 's13', 's14', 's15', 's17', 's20', 's21']


In [34]:
from sklearn.preprocessing import MinMaxScaler
scaler = MinMaxScaler()
train_df[used_sensors] = scaler.fit_transform(train_df[used_sensors])   # train
test_df[used_sensors]  = scaler.transform(test_df[used_sensors])        # test

In [35]:
import numpy as np

def make_window(df, window, sensor_cols):
  X_list, y_list = [], []
  for unit_id in df['unit'].unique():
    eng = df[df['unit'] == unit_id]
    sensors = eng[sensor_cols].values
    ruls = eng['RUL'].values
    for start in range(len(eng) - window + 1):
      X_list.append(sensors[start : start + 30])
      y_list.append(ruls[start + window - 1])
  return np.array(X_list), np.array(y_list)

X, y = make_window(train_df, 30, used_sensors)
print(X.shape, y.shape)

(17731, 30, 15) (17731,)
